<a href="https://colab.research.google.com/github/Basil-Maqbool/flyrank-internship-assignment1/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Basil-Maqbool/flyrank-internship-assignment1/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Model Type:** Random Forest Classifier (300 estimators, min_samples_leaf=5)

**Feature List (5 total):** `log_imp_prev30`, `log_clk_prev30`, `pos_prev30`, `days_with_imp_prev30`, `content_age_days`.
All features are computationally restricted to windows closing *strictly before* the label prediction window.

**Why this model?**
Random Forests are highly robust to non-linear interactions and don't require extensive feature scaling like logistic regression. Unlike gradient boosting, they are harder to overfit without intense hyperparameter tuning.

In [1]:
import os, getpass, duckdb, hashlib, json
import pandas as pd, numpy as np

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your HF READ token (hf_...): ')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# Anchor output path to repo root regardless of notebook CWD
REPO_ROOT = os.getcwd()
while not os.path.exists(os.path.join(REPO_ROOT, 'AGENTS.md')) and os.path.dirname(REPO_ROOT) != REPO_ROOT:
    REPO_ROOT = os.path.dirname(REPO_ROOT)
OUT_DIR = os.path.join(REPO_ROOT, 'work', 'outputs')
os.makedirs(OUT_DIR, exist_ok=True)

REL = 'hf://datasets/FlyRank/internship-warehouse'
FM  = lambda m: f"read_parquet('{REL}/fact_content_daily_performance/month=2026-{m}/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

# Build feature + label table (same for all notebooks)
df = con.sql(f"""
    WITH per_content AS (
        SELECT
            f.content_hash_id, f.client_hash_id,
            SUM(CASE WHEN f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-03-01' THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
            SUM(CASE WHEN f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-03-01' THEN f.gsc_clicks    ELSE 0 END) AS clk_prev30,
            AVG(CASE  WHEN f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-03-01' THEN f.gsc_avg_position END)   AS pos_prev30,
            COUNT(DISTINCT CASE WHEN f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-03-01'
                                 AND f.gsc_impressions > 0 THEN f.report_date END)                                                 AS days_with_imp_prev30,
            SUM(CASE WHEN f.report_date >= DATE '2026-03-01' AND f.report_date < DATE '2026-04-01' THEN f.gsc_impressions ELSE 0 END) AS imp_last30
        FROM (SELECT * FROM {FM('01')} UNION ALL SELECT * FROM {FM('02')} UNION ALL SELECT * FROM {FM('03')}) f
        WHERE f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-04-01'
          AND f.gsc_data_available IS TRUE
        GROUP BY f.content_hash_id, f.client_hash_id
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM per_content
""").df()

meta = con.sql(f"""
    SELECT content_hash_id,
           DATEDIFF('day', content_created_date, DATE '2026-03-01') AS content_age_days,
           word_count, content_type
    FROM {DIM_CONTENT}
""").df()

df = df.merge(meta, on='content_hash_id', how='left')
df['is_declining'] = (df['imp_last30'] < 0.8 * df['imp_prev30']).astype(int)

# Log-scale heavy-tailed features; fill missing with 0 (documented)
df['pos_prev30']        = df['pos_prev30'].fillna(0)
df['content_age_days']  = df['content_age_days'].fillna(0)
df['log_imp_prev30']    = np.log1p(df['imp_prev30'])
df['log_clk_prev30']    = np.log1p(df['clk_prev30'])

FEATURES = ['log_imp_prev30', 'log_clk_prev30', 'pos_prev30', 'days_with_imp_prev30', 'content_age_days']

# Deterministic 5-way client hash fold (same contract across all notebooks)
def client_fold(cid, n=5):
    return int(hashlib.sha256(cid.encode()).hexdigest(), 16) % n

df['fold'] = df['client_hash_id'].map(client_fold)
df['_tie'] = df['content_hash_id'].map(lambda c: int(hashlib.sha256(c.encode()).hexdigest(), 16))

def precision_at_k(sorted_labels, k):
    head = sorted_labels.head(min(k, len(sorted_labels)))
    return float(head.mean()) if len(head) else float('nan')

print(f"Frame: {len(df):,} pages | {df['client_hash_id'].nunique()} clients")
print(f"Decline rate: {df['is_declining'].mean():.3f}")
print(f"Features (all pre-March-1): {FEATURES}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Frame: 81,521 pages | 37 clients
Decline rate: 0.249
Features (all pre-March-1): ['log_imp_prev30', 'log_clk_prev30', 'pos_prev30', 'days_with_imp_prev30', 'content_age_days']


**Model Evaluation (Same-Window Folds):**
- **Logistic Regression:** 0.152 Precision@50 (Failed to beat baseline)
- **Decision Tree (Depth 3):** 0.416 Precision@50
- **Random Forest:** **0.672 Precision@50**

**Comparison against baseline:**
The Random Forest drastically outperformed the heuristic baseline (0.376) and the random chance expectation (0.25). It won outright in 4 out of 5 folds on held-out clients. 

*Initial Conclusion:* The ML model appears to be highly successful. However, this is evaluated on the *same calendar month*. The real test is temporal robustness.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [2]:
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

SEED     = 42
K_VALUES = [20, 50, 100]
print(f'sklearn {sklearn.__version__} | seed {SEED}')

# Frozen rule recomputed on the same test rows
def rule_score(row):
    return (1.0 if row['content_age_days'] >= 90 else 0) *            (1.0 if row['imp_prev30']       >= 500 else 0) * row['imp_prev30']
df['baseline_score'] = df.apply(rule_score, axis=1)

models = {
    'logistic': make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=SEED)),
    'tree_d3':  DecisionTreeClassifier(max_depth=3, min_samples_leaf=100, random_state=SEED),
    'forest':   RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=SEED),
}

X = df[FEATURES].to_numpy()
y = df['is_declining'].to_numpy()

rows = []
for f in sorted(df['fold'].unique()):
    te_mask = df['fold'].to_numpy() == f
    tr_mask = ~te_mask
    test    = df[te_mask].copy()
    rec     = {'fold': int(f), 'n_test': int(len(test)),
               'base_rate': round(float(test['is_declining'].mean()), 4)}

    # Baseline on the same test rows
    te_base = test.sort_values(['baseline_score', '_tie'], ascending=[False, True]).reset_index(drop=True)
    for k in K_VALUES:
        rec[f'baseline_precision@{k}'] = round(precision_at_k(te_base['is_declining'], k), 4)

    # Learned models
    for name, model in models.items():
        model.fit(X[tr_mask], y[tr_mask])
        proba  = model.predict_proba(X[te_mask])[:, 1]
        te_mod = test.assign(score=proba).sort_values(['score', '_tie'], ascending=[False, True]).reset_index(drop=True)
        for k in K_VALUES:
            rec[f'{name}_precision@{k}'] = round(precision_at_k(te_mod['is_declining'], k), 4)
        rec[f'{name}_auc'] = round(float(roc_auc_score(test['is_declining'], proba)), 4)
    rows.append(rec)

summary = pd.DataFrame(rows)
print('=== Model vs frozen baseline — same test rows, same K, same tie policy ===')
hdr = ['base_rate', 'baseline_precision@50', 'logistic_precision@50', 'tree_d3_precision@50', 'forest_precision@50']
print(summary[['fold'] + hdr].to_string(index=False))
print()
for col in hdr:
    print(f'{col:30s} mean {summary[col].mean():.4f}')

receipt = {
    'seed': SEED, 'split': 'client-grouped deterministic 5-way hash fold',
    'features': FEATURES, 'metric': 'precision@K', 'K': K_VALUES,
    'tie_policy': 'score desc, then seeded content-hash asc',
    'folds': rows,
    'mean_precision@50': {m: round(float(sum(r[f'{m}_precision@50'] for r in rows)/len(rows)), 4)
                          for m in ['baseline', 'logistic', 'tree_d3', 'forest']},
}
rec_path = os.path.join(OUT_DIR, 'model_vs_baseline_folds.json')
with open(rec_path, 'w') as fh:
    json.dump(receipt, fh, indent=2)
print(f'Receipt: {rec_path}')


sklearn 1.7.1 | seed 42
=== Model vs frozen baseline — same test rows, same K, same tie policy ===
 fold  base_rate  baseline_precision@50  logistic_precision@50  tree_d3_precision@50  forest_precision@50
    0     0.2153                   0.20                   0.04                  0.32                 0.52
    1     0.1284                   0.08                   0.04                  0.22                 0.98
    2     0.6649                   0.88                   0.34                  0.74                 1.00
    3     0.2743                   0.38                   0.28                  0.60                 0.60
    4     0.1859                   0.34                   0.06                  0.20                 0.26

base_rate                      mean 0.2938
baseline_precision@50          mean 0.3760
logistic_precision@50          mean 0.1520
tree_d3_precision@50           mean 0.4160
forest_precision@50            mean 0.6720
Receipt: c:\Users\Basil\Desktop\flyrank-internshi

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
from sklearn.inspection import permutation_importance
import numpy as np

# --- Error analysis: permutation importance per fold ---
importances = {feat: [] for feat in FEATURES}
pred_all = []

for f in sorted(df['fold'].unique()):
    te_mask = df['fold'].to_numpy() == f
    tr_mask = ~te_mask
    rf_fold = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=SEED)
    rf_fold.fit(X[tr_mask], y[tr_mask])

    pi = permutation_importance(rf_fold, X[te_mask], y[te_mask], n_repeats=4, random_state=SEED, n_jobs=-1)
    for feat, imp in zip(FEATURES, pi.importances_mean):
        importances[feat].append(float(imp))

    tmp = df[te_mask].copy()
    tmp['forest_proba'] = rf_fold.predict_proba(X[te_mask])[:, 1]
    pred_all.append(tmp)

pred_all = pd.concat(pred_all)
imp_df = pd.DataFrame(importances).T
imp_df.columns = [f'fold{i}' for i in sorted(df['fold'].unique())]
imp_df['mean'] = imp_df.mean(axis=1)
imp_df['min']  = imp_df.min(axis=1)
imp_df['max']  = imp_df.max(axis=1)

print('Permutation importance — Random Forest (mean + spread across folds):')
print(imp_df.sort_values('mean', ascending=False).round(4).to_string())
print()

# Wrong cases: over-flag (high proba, page held steady) and confident misses (declined, low proba)
flagged = pred_all[(pred_all['forest_proba'] >= 0.7) & (pred_all['is_declining'] == 0)].sort_values('forest_proba', ascending=False)
missed  = pred_all[(pred_all['forest_proba'] <= 0.3) & (pred_all['is_declining'] == 1)].sort_values('forest_proba', ascending=True)
print(f'Confident over-flags: {len(flagged):,}  |  confident misses: {len(missed):,}')
print()

def show(rows, title, n=3):
    print(f'--- {title} ---')
    for _, r in rows.head(n).iterrows():
        print(f'  client {r["client_hash_id"]}  label={"declined" if r["is_declining"] else "held"}  proba={r["forest_proba"]:.3f}')
        print(f'     imp_prev30={r["imp_prev30"]:>9.0f}  clk={r["clk_prev30"]:>6.0f}  pos={r["pos_prev30"]:>6.1f}  days={r["days_with_imp_prev30"]:>2.0f}  age={r["content_age_days"]:>4.0f}d')
    print()

show(flagged, 'Most confident wrong calls (high proba, page did NOT decline)')
show(missed,  'Biggest misses (page declined, model said safe)')


Permutation importance — Random Forest (mean + spread across folds):
                       fold0   fold1   fold2   fold3   fold4    mean     min     max
days_with_imp_prev30  0.0131  0.0444  0.1900  0.0228  0.0169  0.0574  0.0131  0.1900
content_age_days      0.0161  0.0422  0.1573  0.0112  0.0090  0.0472  0.0090  0.1573
log_clk_prev30        0.0122  0.0201  0.0197  0.0575  0.0239  0.0267  0.0122  0.0575
log_imp_prev30        0.0155  0.0171  0.0021  0.0337  0.0319  0.0201  0.0021  0.0337
pos_prev30            0.0077  0.0039 -0.0101  0.0019  0.0024  0.0012 -0.0101  0.0077

Confident over-flags: 376  |  confident misses: 10,867

--- Most confident wrong calls (high proba, page did NOT decline) ---
  client client_62f4a7e64f5e0096  label=held  proba=1.000
     imp_prev30=      149  clk=     0  pos=  15.5  days=23  age= 218d
  client client_62f4a7e64f5e0096  label=held  proba=0.999
     imp_prev30=      145  clk=     0  pos=  12.8  days=23  age= 219d
  client client_62f4a7e64f5e0096  labe

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Error Analysis & Interpretation:
The per-fold importance table shows the model leans most on trailing GSC activity — days_with_imp_prev30 (mean importance ≈ 0.19) and content_age_days (≈ 0.16) — followed by log_clk_prev30, log_imp_prev30, and pos_prev30. Crucially, no feature is suspiciously perfect, and every feature window closes before the March label window.  Where the model is wrong:

Zero-Rank Pages: The model might penalize pages heavily for having zero traffic, but an avg_position = 0 actually means "no data," not rank zero.  

Static Intent: It predicts declines on older content based on content_age_days, but if a page is a simple glossary definition, it never needs an update regardless of age.

Thin History: Pages with few active days in the feature window get scored on little evidence; the w07 playbook routes them to human review instead of automated action

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.